# CH7 互動式視覺化與 Plotly 基礎

## 7.1 互動式視覺化簡介

互動式視覺化（Interactive Visualization）是指使用者可以與圖表即時互動，透過滑鼠懸停查看細節、縮放放大／平移圖表區域，以及點選圖例開關不同資料系列。

- **優勢**：
  - **探索性分析**：即時篩選與放大，快速找出資料異常點。
  - **深度洞察**：滑鼠懸停顯示更多維度資訊，不必額外標註。
  - **動態呈現**：動畫或切換工具，更易吸引互動與討論。

- **應用場景（醫療領域）**：
  - 病患生理指標探索（血糖、血壓、BMI）。
  - 基因表現量與臨床特徵關係。
  - 醫療影像標註區域統計。
  - 疫情趨勢與地理熱點圖。

## 7.2 Plotly 詳細介紹與支援圖表類型

**Plotly** 是一個功能強大的 Python 視覺化套件，提供豐富的 API，支援在瀏覽器上動態呈現互動式圖表。

- **核心套件**：
  - `plotly.express`：高階介面，語法簡潔，適合快速繪圖。
  - `plotly.graph_objects`：低階介面，完全控制 Trace、Layout，適合複雜自訂需求。

- **安裝**：
  ```bash
  pip install plotly
  ```

- **主要物件**：
  - **Figure**：整張圖表容器。
  - **Trace**：一組資料系列（如一條折線、一組散點）。
  - **Layout**：整體佈局設定（標題、軸標籤、主題樣式）。

- **支援的常用圖表類型**：
  1. 折線圖（Line Chart）
  2. 長條圖（Bar Chart）
  3. 散點圖（Scatter Plot）
  4. 面積圖（Area Chart）
  5. 箱型圖（Box Plot）
  6. 小提琴圖（Violin Plot）
  7. 直方圖（Histogram）
  8. 密度圖（Density Plot）
  9. 熱力圖（Heatmap）
  10. 地理圖（Choropleth）……等

## 7.3 基礎範例實作（以 Pima 糖尿病資料集為例）

首先載入資料並做初步檢視：

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
path = '/content/drive/MyDrive/Colab Notebooks/CH7_dataset/datasets'

Mounted at /content/drive


In [3]:
import pandas as pd
from pathlib import Path

data_file = "diabetes.csv"

df = pd.read_csv(path+"/"+data_file)


# Display summary: number of rows, columns, non-null count
df.info()

# Preview first 5 rows to check column names and values
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


### 7.3.1 散點圖 (Scatter Plot)

**分析目標**：探索病人年齡 (Age) 與血糖值 (Glucose) 的關係，並依是否罹患糖尿病 (Outcome) 分色顯示。

**圖表類型**：散點圖  
**互動功能**：hover 顯示 BMI、BloodPressure；zoom & pan；legend 切換系列

In [4]:
import plotly.express as px

# 使用 Plotly Express 繪製散點圖
fig = px.scatter(
    df,
    x='Age',                 # x 軸：年齡
    y='Glucose',             # y 軸：血糖濃度
    color='Outcome',         # 依 Outcome 分色 (0=否,1=是)
    title='年齡 vs 血糖濃度（依糖尿病分群）',
    labels={'Age':'年齡','Glucose':'血糖濃度'},
    hover_data=['BMI', 'BloodPressure']  # hover 顯示更多欄位
)
fig.update_layout(legend_title_text='是否糖尿病')  # 設定圖例標題
fig.show()

### Try it!：7.3.1 散點圖 (Scatter Plot)
想觀察 BMI 與胰島素之間的關係，並區分是否罹患糖尿病。

**程式要求**
1. 使用 Plotly Express 繪製 `BMI`（x 軸）對 `Insulin`（y 軸）的散點圖。
2. 以 `Outcome` 分色，hover 顯示 `Age`、`Glucose`、`BloodPressure`。
3. 加入 `trendline='ols'` 迴歸趨勢線並顯示圖表。

### 7.3.2 長條圖 (Bar Chart)

**分析目標**：比較不同年齡組的糖尿病人數分布。

**圖表類型**：分組長條圖  
**互動功能**：hover 顯示人數；zoom & pan；legend 切換系列

In [5]:
# Create age group column if this cell is run independently.
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[20,30,40,50,60,70,80],
    labels=['20-29','30-39','40-49','50-59','60-69','70-79']
)

fig = px.bar(
    df.groupby(['AgeGroup','Outcome'], observed=False).size().reset_index(name='Count'),
    x='AgeGroup', y='Count', color='Outcome', barmode='group',
    title='不同年齡段的糖尿病人數分布', labels={'AgeGroup':'年齡組','Count':'人數'}
)
fig.update_traces(hovertemplate='年齡組：%{x}<br>人數：%{y}<extra></extra>')
fig.update_layout(legend_title_text='是否糖尿病')
fig.show()

### Try it!：7.3.2 長條圖 (Bar Chart)
範例統計的是各年齡層「人數」，這次改看各年齡層的「平均血糖」在罹病與未罹病兩組之間的差異。

**程式要求**
1. 建立 `AgeGroup` 欄位（每 10 歲一組，`20-29` 到 `70-79`）。
2. 計算各 `AgeGroup` 與 `Outcome` 組合的平均 `Glucose` 並印出。
3. 使用 Plotly 繪製群組長條圖（`barmode='group'`）並顯示。

### 7.3.3 折線圖 (Line Chart)

**分析目標**：顯示隨年齡變化的平均 BMI 趨勢。

**圖表類型**：折線圖  
**互動功能**：hover 顯示 BMI；zoom & pan；legend（單系列可隱藏）

In [6]:
fig = px.line(
    df.groupby('Age')['BMI'].mean().reset_index(),
    x='Age', y='BMI',
    title='不同年齡的平均 BMI 趨勢', labels={'Age':'年齡','BMI':'平均 BMI'}
)
fig.update_layout(hovermode='x unified')
fig.show()

### Try it!：7.3.3 折線圖 (Line Chart)
觀察 BMI 與血糖隨年齡變化的趨勢。

**程式要求**
1. 依 `Age` 分組，計算平均 `BMI` 與平均 `Glucose`。
2. 使用 Plotly 繪製折線圖並開啟 `markers=True`，兩條折線放在同一張圖。
3. 設定 `hovermode='x unified'` 後顯示圖表。

### 7.3.4 面積圖 (Area Chart)

**分析目標**：展示隨年齡累積的糖尿病病例（Outcome=1）。

**圖表類型**：面積圖  
**互動功能**：hover 顯示累積病例數；zoom & pan；legend 控制層級

In [7]:
age_case = df[df['Outcome']==1].groupby('Age').size().cumsum().reset_index(name='CumulativeCases')
fig = px.area(
    age_case, x='Age', y='CumulativeCases',
    title='隨年齡累積糖尿病病例數', labels={'Age':'年齡','CumulativeCases':'累積病例數'}
)
fig.update_traces(hovertemplate='年齡：%{x}<br>累積病例數：%{y}<extra></extra>')
fig.show()

### Try it!：7.3.4 面積圖 (Area Chart)
範例只累積罹病病例，這次把罹病與未罹病兩組的累積人數畫在同一張面積圖比較成長速度。

**程式要求**
1. 依 `Age` 與 `Outcome` 分組計算人數，並在各組內對年齡做累加。
2. 使用 Plotly 繪製面積圖，以 `Outcome` 分色呈現兩塊面積。
3. 顯示圖表。

### 7.3.5 箱型圖 (Box Plot)

**分析目標**：比較糖尿病與非糖尿病組別的 BMI 分布情況。

**圖表類型**：箱型圖  
**互動功能**：hover 顯示 Q1、Median、Q3；zoom & pan；legend 切換

In [8]:
fig = px.box(
    df, x='Outcome', y='BMI',
    title='糖尿病 vs 非糖尿病 的 BMI 分布比較', labels={'Outcome':'是否糖尿病','BMI':'BMI'}
)
fig.update_traces(hovertemplate='組別：%{x}<br>Q1：%{q1}<br>中位數：%{median}<br>Q3：%{q3}<extra></extra>')
fig.show()

### Try it!：7.3.5 箱型圖 (Box Plot)
比較罹病與未罹病族群的 BMI 與血糖分布。

**程式要求**
1. 使用 Plotly 依 `Outcome` 繪製 `BMI` 箱型圖。
2. 再繪製一張 `Glucose` 箱型圖。
3. 兩張圖都直接顯示。

### 7.3.6 小提琴圖 (Violin Plot)

**分析目標**：更細緻地比較兩組 BMI 密度分布與原始資料點。

**圖表類型**：小提琴圖  
**互動功能**：hover 顯示密度值；zoom & pan；legend 分組切換

In [9]:
fig = px.violin(
    df, x='Outcome', y='BMI', box=True, points='all',
    title='BMI 分布之小提琴圖（含箱型與資料點）', labels={'Outcome':'是否糖尿病','BMI':'BMI'}
)
fig.update_traces(hovertemplate='組別：%{x}<br>BMI：%{y}<extra></extra>')
fig.show()

### Try it!：7.3.6 小提琴圖 (Violin Plot)
範例看的是 BMI，這次改用血糖，檢查兩組的分布差異是否更明顯。

**程式要求**
1. 使用 Plotly 依 `Outcome` 繪製 `Glucose` 小提琴圖。
2. 開啟內嵌箱型圖（`box=True`）與所有資料點（`points='all'`）。
3. 顯示圖表。

### 7.3.7 直方圖 (Histogram) + 密度圖 (Density Plot)

**分析目標**：檢視血糖 (Glucose) 的分布型態，並加上核密度估計曲線。

**圖表類型**：直方圖 + 密度曲線  
**互動功能**：hover 顯示頻數／密度；zoom & pan；legend 切換

In [10]:
from plotly.figure_factory import create_distplot
hist_data = [df['Glucose']]
group_labels = ['Glucose']
fig = create_distplot(hist_data, group_labels, bin_size=10, curve_type='kde')
fig.update_layout(title='血糖分布直方圖與密度估計', xaxis_title='血糖濃度', yaxis_title='頻率 / 密度')
fig.show()

### Try it!：7.3.7 直方圖 (Histogram) + 密度圖 (Density Plot)
觀察血糖數值的分布，並比較罹病與未罹病兩組的差異。

**程式要求**
1. 依 `Outcome` 將 `Glucose` 分成「未罹病」與「罹病」兩組。
2. 像範例一樣使用 `create_distplot` 繪製直方圖與密度曲線，保留下方的條狀分布（rug）。
3. 以中文設定標題與 x、y 軸標籤後顯示圖表。

## 7.4 互動功能重點整理

- **hover**：透過 `hover_data`（Express）或 `hovertemplate`（GO）自訂懸停資訊。
- **zoom & pan**：Plotly 圖表預設支援，使用滑鼠滾輪放大／拖曳平移。
- **legend**：透過 `update_layout(legend_title='...')` 管理圖例，點擊切換系列。

## 7.5 課堂練習（共 5 題，含解答範例）

每題請明確說明**分析目標**、**圖表類型**與**互動功能**，並提供可執行的程式碼。

### 題目 1

**分析目標**：探索胰島素 (Insulin) 與 BMI 的關係，分群顯示是否糖尿病。

**圖表類型**：散點圖  
**互動功能**：hover 顯示 Age、Glucose；zoom & pan；legend 切換

In [11]:
fig = px.scatter(
    df, x='Insulin', y='BMI', color='Outcome',
    title='胰島素 vs BMI（依糖尿病分群）', labels={'Insulin':'胰島素','BMI':'BMI'},
    hover_data=['Age','Glucose']
)
fig.update_layout(legend_title_text='是否糖尿病')
fig.show()

### Try it!：題目 1
綜合練習：血糖與胰島素的關係（與範例解答使用不同的變數組合）。

**程式要求**
1. 使用 Plotly Express 繪製 `Glucose`（x 軸）對 `Insulin`（y 軸）的散點圖，以 `Outcome` 分色。
2. hover 顯示 `Age`、`BMI`、`BloodPressure`。
3. 顯示圖表。

### 題目 2

**分析目標**：比較不同 BMI 分組（<25、25–30、>30）之糖尿病人數比例。

**圖表類型**：分組長條圖  
**互動功能**：hover 顯示比例；zoom & pan；legend 切換

In [12]:
df['BMICat'] = pd.cut(df['BMI'], bins=[0,25,30,100], labels=['<25','25-30','>30'])
bmi_count = df.groupby(['BMICat','Outcome'], observed=False).size().reset_index(name='Count')
total = bmi_count.groupby('BMICat', observed=False)['Count'].transform('sum')
bmi_count['Ratio'] = bmi_count['Count']/total
fig = px.bar(bmi_count, x='BMICat', y='Ratio', color='Outcome', barmode='group',
             title='不同 BMI 分組的糖尿病比例', labels={'BMICat':'BMI 分組','Ratio':'比例'})
fig.update_traces(hovertemplate='BMI 組：%{x}<br>比例：%{y:.2%}<extra></extra>')
fig.update_layout(legend_title_text='是否糖尿病')
fig.show()

### Try it!：題目 2
綜合練習：改用 WHO 四分類檢視 BMI 與罹病比例的關係。

**程式要求**
1. 建立 `BMICat` 欄位，分為 `Underweight`（<18.5）、`Normal`（18.5-25）、`Overweight`（25-30）、`Obese`（>30）四組。
2. 計算各組中罹病與未罹病的人數與比例並印出。
3. 使用 Plotly 繪製群組長條圖（y 軸為比例，hover 顯示人數）並顯示。

### 題目 3

**分析目標**：顯示不同年齡組（每 10 歲一組）平均血壓（BloodPressure）趨勢。

**圖表類型**：折線圖  
**互動功能**：hover 顯示平均血壓；zoom & pan；legend（單系列可隱藏）

In [13]:
df['AgeDecade'] = pd.cut(df['Age'], bins=range(20,91,10), right=False,
                           labels=[f'{i}-{i+9}' for i in range(20,90,10)])
bp_mean = df.groupby('AgeDecade', observed=False)['BloodPressure'].mean().reset_index()
fig = px.line(bp_mean, x='AgeDecade', y='BloodPressure',
               title='不同年齡組的平均血壓趨勢', labels={'AgeDecade':'年齡組','BloodPressure':'平均血壓'})
fig.update_layout(hovermode='x unified')
fig.show()

### Try it!：題目 3
綜合練習：資料中 `BloodPressure = 0` 其實是缺測值，先排除再看各年齡層的平均血壓。

**程式要求**
1. 排除 `BloodPressure == 0` 的資料列，並印出排除的筆數。
2. 將 `Age` 以每 10 歲分組建立 `AgeDecade`，計算各組平均 `BloodPressure` 與樣本數並印出。
3. 使用 Plotly 繪製折線圖（`markers=True`，hover 顯示樣本數）並顯示。

### 題目 4

**分析目標**：檢視血糖 (Glucose) 的分布型態，繪製直方圖與密度估計。

**圖表類型**：直方圖 + 密度曲線  
**互動功能**：hover 顯示頻數／密度；zoom & pan；legend 切換

In [14]:
fig = create_distplot([df['Glucose']], ['Glucose'], bin_size=10, curve_type='kde')
fig.update_layout(title='血糖分布直方圖與密度估計', xaxis_title='血糖濃度',
                  yaxis_title='頻率 / 密度')
fig.show()

### Try it!：題目 4
綜合練習：改看血壓的分布（與範例解答使用不同的變數）。

**程式要求**
1. 使用 Plotly 繪製 `BloodPressure` 直方圖。
2. 以 `Outcome` 分色並設定 `barmode='overlay'` 與透明度。
3. 顯示圖表。

### 題目 5

**分析目標**：比較糖尿病與非糖尿病組別 BMI 的箱型圖與小提琴圖。

**圖表類型**：箱型圖 + 小提琴圖  
**互動功能**：hover 顯示分位數與密度；zoom & pan；legend 切換

In [15]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Box(y=df[df['Outcome']==0]['BMI'], name='非糖尿病', boxpoints='outliers',
                hovertemplate='Q1：%{q1}<br>中位數：%{median}<br>Q3：%{q3}<extra></extra>'))
fig.add_trace(go.Box(y=df[df['Outcome']==1]['BMI'], name='糖尿病', boxpoints='outliers',
                hovertemplate='Q1：%{q1}<br>中位數：%{median}<br>Q3：%{q3}<extra></extra>'))
fig.add_trace(go.Violin(y=df[df['Outcome']==0]['BMI'], name='非糖尿病 (Violin)', side='negative', points='all', hoverinfo='y+name'))
fig.add_trace(go.Violin(y=df[df['Outcome']==1]['BMI'], name='糖尿病 (Violin)', side='positive', points='all', hoverinfo='y+name'))
fig.update_layout(title='BMI 箱型圖與小提琴圖比較', yaxis_title='BMI', legend_title='組別')
fig.show()

### Try it!：題目 5
綜合練習：箱型圖與小提琴圖並用，觀察胰島素分布（與範例解答使用不同的變數）。

**程式要求**
1. 使用 Plotly 依 `Outcome` 繪製 `Insulin` 箱型圖（`points='all'`）。
2. 再依 `Outcome` 繪製 `Insulin` 小提琴圖（`box=True`、`points='all'`）。
3. 兩張圖都直接顯示。